# Многопоточность
Многопоточность - концепция программирования, которая позволяет выполнять несколько потоков в рамках одного процесса. 

При этом надо понимать, что одно ядро не ограничено одним процессом.

In [16]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor

In [17]:


def worker(name, delay):
    print(f"{name}: начал работу\n", end="")
    time.sleep(delay)  # Блокирующая операция (имитация долгой работы)
    print(f"{name}: закончил через {delay}с")

# Создаем и запускаем потоки
thread1 = threading.Thread(target=worker, args=("Поток 1", 3))
thread2 = threading.Thread(target=worker, args=("Поток 2", 2))
thread3 = threading.Thread(target=worker, args=("Поток 3", 1))

# Запускаем потоки
thread1.start()
thread2.start()
thread3.start()

# Ждем завершения всех потоков
thread3.join()
thread2.join()
thread1.join()

print("Все потоки завершены")

Поток 1: начал работу
Поток 2: начал работу
Поток 3: начал работу
Поток 3: закончил через 1с
Поток 2: закончил через 2с
Поток 1: закончил через 3с
Все потоки завершены


In [18]:
def worker(name_delay):
    name, delay = name_delay
    print(f"{name}: начал работу\n", end="")
    time.sleep(delay)  # Блокирующая операция
    print(f"{name}: закончил через {delay}с")
    return f"{name} завершен"

# Данные для задач
tasks = [
    ("Поток 1", 3),
    ("Поток 2", 2),
    ("Поток 3", 1)
]

# Запускаем через ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(worker, tasks)

# Результаты (если нужны)
for result in results:
    print(result)

print("Все потоки завершены")

Поток 1: начал работу
Поток 2: начал работу
Поток 3: начал работу
Поток 3: закончил через 1с
Поток 2: закончил через 2с
Поток 1: закончил через 3с
Поток 1 завершен
Поток 2 завершен
Поток 3 завершен
Все потоки завершены


Разница заключается лишь в удобстве использования, так как ThreadPoolExecutor автоматически регулирует работу с потоками.

## ⚠️ GIL (Global Interpreter Lock) — главное ограничение

GIL — это глобальная блокировка интерпретатора, которая позволяет только одному потоку выполнять Python-код в любой момент времени.

Представьте, что это один ключ от общей комнаты, где лежит Python. В комнате может находиться только один человек (поток) с ключом. Остальные ждут снаружи.

In [19]:
# Поток 1
def task_1():
    total = 0
    for i in range(10_000_000):
        total += i  # ← ЕСТЬ GIL (ключ у этого потока)
    return total

# Поток 2
def task_2():
    total = 0
    for i in range(10_000_000):
        total += i * i  # ← НЕТ GIL (ждет ключ)

# Запускаем два потока
t1 = threading.Thread(target=task_1)
t2 = threading.Thread(target=task_2)

t1.start()
t2.start()
t1.join()
t2.join()

# На самом деле они НЕ выполняются параллельно!
# Сначала работает один поток, потом другой

Это приводит нас к тому, что CPU-интенсивные процессы по сути не экономят время. В то время как I/O-интенсивные задачи не страдают